# Analytic Method: CS-01 TAS

**Purpose**: solve the Tele Assistance System as an open Jackson queueing network in closed form (M/M/c/K per node) across the full adaptation axis and check it against the Camara 2023 R1 / R2 / R3 targets.

**Inputs**: `data/config/profile/{dflt,opti}.json` (PACS-style Variable dicts) and `data/reference/baseline.json` (R1 / R2 / R3 thresholds).

**Outputs**:
- `data/results/analytic/<adaptation>/<profile>.json` - per-run metrics (nodes + network + routing + lambda_z).
- `data/results/analytic/<adaptation>/requirements.json` - R1 / R2 / R3 verdicts.
- `data/img/analytic/<adaptation>/*.png` - topology, heatmap, bars, delta figures.

**Equivalent CLI** (reproduces the four runs written by this notebook):
```bash
python -m src.methods.analytic --adaptation baseline
python -m src.methods.analytic --adaptation s1
python -m src.methods.analytic --adaptation s2
python -m src.methods.analytic --adaptation aggregate
```

This notebook is thin: all logic lives in `src.methods.analytic` and `src.view.qn_diagram`. The cells below just orchestrate, display, and save figures.

In [ ]:
%matplotlib inline
from pathlib import Path

import pandas as pd

from src.methods.analytic import run
from src.view import (
    plot_qn_topology,
    plot_node_heatmap,
    plot_node_diffmap,
    plot_arch_bars,
    plot_arch_delta,
)

IMG_ROOT = Path("data/img/analytic")
ADAPTATIONS = ["baseline", "s1", "s2", "aggregate"]

# Human-readable scenario labels for plot titles / legends.
DISPLAY = {
    "baseline": "No Adaptation",
    "s1": "S1: Retry",
    "s2": "S2: Select-Reliable",
    "aggregate": "S1 & S2",
}

## 1. Solve every adaptation

`run(adp=a, wrt=True)` loads the resolved `NetCfg`, solves the Jackson network, writes the per-run envelope under `data/results/analytic/<adaptation>/`, and returns the per-node + aggregate DataFrames plus the R1 / R2 / R3 verdict.

In [ ]:
results = {a: run(adp=a, wrt=True) for a in ADAPTATIONS}

# unpack the per-adaptation pieces used in every later cell
cfgs = {a: results[a]["config"] for a in ADAPTATIONS}
nodes = {a: results[a]["nodes"] for a in ADAPTATIONS}
nets = {a: results[a]["network"] for a in ADAPTATIONS}
reqs = {a: results[a]["requirements"] for a in ADAPTATIONS}

print(f"Solved {len(results)} adaptations; wrote JSONs under data/results/analytic/")

## 2. Network-wide summary + verdict

One row per adaptation: headline metrics (response time, utilisation, queue length) and the R1 / R2 / R3 pass flags.

In [ ]:
rows = []
for a in ADAPTATIONS:
    n = nets[a].iloc[0]
    r = reqs[a]
    rows.append({
        "adaptation": a,
        "profile": cfgs[a].profile,
        "W_net (ms)": n["W_net"] * 1000,
        "avg_rho": n["avg_rho"],
        "max_rho": n["max_rho"],
        "L_net": n["L_net"],
        "R1": "PASS" if r["R1"]["pass"] else "FAIL",
        "R2": "PASS" if r["R2"]["pass"] else "FAIL",
        "R3": "PASS" if r["R3"]["pass"] else "FAIL",
    })
summary = pd.DataFrame(rows).set_index("adaptation")
summary.round(4)

## 3. Per-node snapshot (baseline)

The reference configuration before any adaptation kicks in. `rho = lambda / (c * mu)`; `W`, `Wq` are in seconds; `L`, `Lq` are mean request counts.

In [ ]:
nodes["baseline"][[
    "key", "name", "type", "lambda", "mu", "c", "K",
    "rho", "L", "W",
]].round(4)

## 4. Queue-network topology (architecture view)

Nodes are coloured by `rho` (cool = low, warm = high). Edge labels show routing probabilities.

In [ ]:
# standalone topology for every adaptation -> data/img/analytic/<adp>/topology.png
for a in ADAPTATIONS:
    plot_qn_topology(
        routs=cfgs[a].routing,
        ndss=nodes[a],
        nets=nets[a],
        title=f"Analytic: QN topology\n{DISPLAY[a]}",
        file_path=str(IMG_ROOT / a),
        fname="topology.png")

## 5. Per-node heatmap (before vs after)

Each row = one artifact; each column = one metric. Columns are normalised per-metric across both scenarios so the heat value is directly comparable. Numeric cell labels show the raw value.

In [ ]:
node_keys = nodes["baseline"]["key"].tolist()
heat_metrics = ["rho", "L", "Lq", "W", "Wq"]
heat_labels = [
    r"$\rho$",
    r"$L$ [req]",
    r"$L_q$ [req]",
    r"$W$ [s/req]",
    r"$W_q$ [s/req]",
]

for a in ["s1", "s2", "aggregate"]:
    plot_node_heatmap(
        ndss=[nodes["baseline"], nodes[a]],
        names=[DISPLAY["baseline"], DISPLAY[a]],
        nodes=node_keys,
        metrics=heat_metrics,
        labels=heat_labels,
        title=f"Analytic: Per-comp metrics heatmap\n{DISPLAY[a]} vs {DISPLAY['baseline']}",
        file_path=str(IMG_ROOT / a),
        fname="heatmap_vs_baseline.png")

## 5b. Per-node delta heatmap (Δ% vs baseline)

Single-panel view of the per-node, per-metric percent change between baseline and each adaptation. Centred diverging colour scale so positive and negative changes read as equal-intensity colours.

In [ ]:
# single-panel delta heatmap per adaptation -> data/img/analytic/<adp>/nd_diffmap_vs_baseline.png
# Match the OLD convention: delta = (opti - dflt) / |dflt| -- ratio, not percent.
diff_metrics = ["rho", "L", "Lq", "W", "Wq"]
diff_labels = [
    r"$\Delta \rho$",
    r"$\Delta L$",
    r"$\Delta L_q$",
    r"$\Delta W$",
    r"$\Delta W_q$",
]

bl_nodes = nodes["baseline"]
for a in ["s1", "s2", "aggregate"]:
    ac_nodes = nodes[a]
    rows = []
    # align by positional index (slot) since `key` may differ at swap slots
    for i in range(len(ac_nodes)):
        b_row = bl_nodes.iloc[i]
        c_row = ac_nodes.iloc[i]
        row = {"key": c_row["key"]}
        for m in diff_metrics:
            b, c = float(b_row[m]), float(c_row[m])
            row[m] = ((c - b) / abs(b)) if b else 0.0
        rows.append(row)
    deltas = pd.DataFrame(rows)

    plot_node_diffmap(
        deltas=deltas,
        nodes=deltas["key"].tolist(),
        metrics=diff_metrics,
        labels=diff_labels,
        title=f"Analytic: Per-comp metrics shift\n{DISPLAY[a]} vs {DISPLAY['baseline']}",
        file_path=str(IMG_ROOT / a),
        fname="nd_diffmap_vs_baseline.png")

## 6. Network-wide bars (all four adaptations)

Headline comparison of the four configurations on the metrics that drive the R1 / R2 / R3 verdicts. Y-axis is log-scaled because the metrics span several orders of magnitude (W in seconds vs L in requests).

In [ ]:
bar_metrics = ["W_net", "avg_rho", "max_rho", "L_net"]
bar_labels = [
    r"$\mathbf{\hat{W}_{TAS}}$ [s/req]",
    r"$\mathbf{\hat{\rho}_{TAS}}$ [n.a.]",
    r"$\mathbf{\rho_{TAS,\,Max}}$ [n.a.]",
    r"$\mathbf{\hat{L}_{TAS}}$ [req]",
]

plot_arch_bars(
    nets=[nets[a] for a in ADAPTATIONS],
    names=ADAPTATIONS,
    metrics=bar_metrics,
    labels=bar_labels,
    title="Analytic: TAS Architecture metrics across adaptations",
    file_path=str(IMG_ROOT / "aggregate"),
    fname="net_bars_all.png")

## 7. Network-wide delta (% change vs baseline)

For every non-baseline adaptation: fractional change on the headline metrics. Green bars = improvement, red = degradation. `total_throughput` flips the sign convention (more is better).

In [ ]:
# fractional delta vs baseline for each non-baseline adaptation.
# Metric set + LaTeX labels mirror the reference figure
# __OLD__/data/results/cs1/img/net_analytical_metric_differences.png.
delta_metrics = [
    "avg_mu",
    "avg_rho",
    "L_net",
    "Lq_net",
    "W_net",
    "Wq_net",
    "total_throughput",
]

delta_labels = [
    r"$\mathbf{\Delta \overline{\mu}}$ [req/s]",
    r"$\mathbf{\Delta \overline{\rho}}$ [n.a.]",
    r"$\mathbf{\Delta L_{net}}$ [req]",
    r"$\mathbf{\Delta L_{q,net}}$ [req]",
    r"$\mathbf{\Delta \overline{W}_{net}}$ [s/req]",
    r"$\mathbf{\Delta \overline{W}_{q,net}}$ [s/req]",
    r"$\mathbf{\Delta Th_{net}}$ [req/s]",
]

bl = nets["baseline"].iloc[0]
for a in ["s1", "s2", "aggregate"]:
    ac = nets[a].iloc[0]
    row = {
        m: (ac[m] - bl[m]) / bl[m] if bl[m] else 0.0
        for m in delta_metrics
    }
    plot_arch_delta(
        deltas=pd.DataFrame([row]),
        metrics=delta_metrics,
        labels=delta_labels,
        title=f"Analytic: TAS Architecture metrics shift\n{DISPLAY[a]} vs {DISPLAY['baseline']}",
        file_path=str(IMG_ROOT / a),
        fname="net_delta_vs_baseline.png")

## 8. R1 / R2 / R3 verdict table

Thresholds come from [`data/reference/baseline.json`](data/reference/baseline.json):

- **R1** Availability: `fail_rate <= 0.03 %` (fraction `0.0003`)
- **R2** Performance: `resp_time <= 26 ms`
- **R3** Minimise `cost` subject to `R1 and R2`

`R3` has no numeric threshold; it passes whenever both `R1` and `R2` do.

In [ ]:
req_rows = []
for a in ADAPTATIONS:
    r = reqs[a]
    req_rows.append({
        "adaptation": a,
        "R1 fail_rate": r["R1"]["value"],
        "R1 pass": r["R1"]["pass"],
        "R2 resp_time (s)": r["R2"]["value"],
        "R2 pass": r["R2"]["pass"],
        "R3 pass": r["R3"]["pass"],
    })
pd.DataFrame(req_rows).set_index("adaptation")

## Summary

At the nominal 345 req/s arrival rate, all four adaptations pass R1, R2, and R3. The aggregate configuration (opti routing + opti services) delivers the lowest `W_net` and the lowest `max_rho`; s1 alone (opti routing, dflt services at the swap slots) is the worst on `max_rho` because opti routing pushes more load into dflt services.

**Next method in the pipeline**: `stochastic.ipynb` (SimPy DES) produces the ground-truth distribution this closed-form solution is compared against.